In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import cv2

# Extracted data Directory
input_data_dir = "/media/yshen47/674eaf4c-ac33-42a8-9a4c-cace68d8cccd/home/gem/demo_ws/src/HIL_Research/src/hil/data/cone_il_extracted/cone_first_pass_2022-12-15-09-07-15"

# topic_save_paths = [
#             "pc", 
#             "gps.csv", "accel.csv", "brake.csv", "steer.csv", "speed.csv",
#             "depth", "img_l", "img_r"
#         ]
# Load the paths to each csv/data directory
imgl_format_path = os.path.join(input_data_dir, "img_l", "{}.png")
depth_format_path = os.path.join(input_data_dir, "depth", "{}.npy")

gps_path = os.path.join(input_data_dir, "gps.csv")
brake_path = os.path.join(input_data_dir, "brake.csv")
accel_path = os.path.join(input_data_dir, "accel.csv")
steer_path = os.path.join(input_data_dir, "steer.csv")
speed_path = os.path.join(input_data_dir, "speed.csv")

# Holds the timestamps/sequence numbers for synced sets of img/gps/pacmod data, 
# Each row in this csv is one synced datapoint 
sync_csv_path = os.path.join(input_data_dir, "synced.csv")
df = pd.read_csv(sync_csv_path)
df

For all Pacmod data(brake, acceleratation, steer, speed), the `synced.csv` file contains the relevant sequence number for the respective pacmod data csv to pull for each data point.

In [ ]:
df_brake = pd.read_csv(brake_path)
# print(df_brake.head())
brake_vals = []
for i in df["brake_seqnum"]:
    # match sequence number from synced.csv to a row in brake.csv and append the brake command value in the row(field.f64_cmd)
    brake_vals.append(df_brake[df_brake["field.header.seq"]==i]["field.f64_cmd"].item())
brake_vals = np.array(brake_vals)

In [ ]:
df_accel = pd.read_csv(accel_path)
accel_vals = []
for i in df["accel_seqnum"]:
    accel_vals.append(df_accel[df_accel["field.header.seq"]==i]["field.f64_cmd"].item())
accel_vals = np.array(accel_vals)

In [ ]:
df_steer = pd.read_csv(steer_path)
steer_vals = []
for i in df["steer_seqnum"]:
    steer_vals.append(df_steer[df_steer["field.header.seq"]==i]["field.manual_input"].item())
steer_vals = np.array(steer_vals)

In [ ]:
df_speed = pd.read_csv(speed_path)
speed_vals = []
for i in df["speed_seqnum"]:
    speed_vals.append(df_speed[df_speed["field.header.seq"]==i]["field.vehicle_speed"].item())
speed_vals = np.array(speed_vals)

In [ ]:
plt.plot(brake_vals, label="brake")
plt.plot(accel_vals, label="accel")
plt.plot(speed_vals, label="speed")
plt.plot(steer_vals, label="steer")
plt.title("Visualize synced pacmod values over time")
plt.legend()
plt.show()

In [ ]:
df_gps = pd.read_csv(gps_path)
gps_vals = []
for i in df["gps_stamp"]:
    matched_row = df_gps[df_gps["field.header.stamp"]==i]
    fields = ["field.latitude", "field.longitude", "field.height", "field.roll", "field.pitch", "field.azimuth", "field.header.stamp"]
    data_pt = np.array([
        matched_row[field].item() for field in fields
    ])
    gps_vals.append(data_pt)
gps_vals = np.array(gps_vals)

In [ ]:
imgl_paths = [imgl_format_path.format(stamp) for stamp in df['image_stamp']]
img_vals = []
for i in imgl_paths:
    img = cv2.imread(i)
    img_vals.append(img)
    # cv2.imshow("res", img)
    # cv2.waitKey(5)
img_vals = np.array(img_vals)

In [ ]:
depth_paths = [depth_format_path.format(stamp) for stamp in df['depth_stamp']]
depth_vals = []
for path in depth_paths:
    depth_vals.append(np.load(path))
depth_vals = np.array(depth_vals)

All synced data arrays for this extracted rosbag

In [ ]:
print(gps_vals.shape)
print(speed_vals.shape)
print(steer_vals.shape)
print(accel_vals.shape)
print(brake_vals.shape)
print(depth_vals.shape)
print(img_vals.shape)